# Transiciones largas · comparación con módulo 1
Fuentes de 18–35 → 60/65/70, sin foto objetivo y sin observaciones conocidas a ±10 años. Todas las salidas quedan bajo `OUTPUT`.

### 1. Rutas de Colab

In [ ]:
from pathlib import Path

TRAIN_ROOT = Path("/content/drive/MyDrive/dataset_def/train")
REFERENCE_DIR = Path("/content/drive/MyDrive/aging_bias_references_v2")
CHECKPOINT_PATH = Path("/content/drive/MyDrive/adapter_inference.pt")
OUTPUT = Path("/content/drive/MyDrive/paper_long_gap_counterfactual_v1")

### 2. Configuración

In [ ]:
MODELS = ["SAM", "FRAN", "FADING", "HRFAE",
          "ours_strength_0.31", "ours_picking_no_prompt", "ours_picking_prompt"]

CONFIG = {
    "n_transitions": 350,
    "source_ages": (18, 35),
    "target_ages": (60, 65, 70),
    "exclusion_margin": 10,
    "batch_size": 32,
    "n_boot": 1000,
    "run_dino": True,
    "gender_csv_path": "/content/genero_por_id.csv",
    "model_loader_config": {
        "root": "/content/aging_benchmarks",
        "fading_specialized_path": globals().get("FADING_SPECIALIZED_PATH"),
    },
}

### 3. Seleccionar, generar y guardar
Reanuda lo ya generado. Conserva los batches de los modelos externos y la inferencia normal/picking de nuestros modelos.

In [ ]:
# Preparacion de Colab dentro de esta misma celda.
try:
    from google.colab import drive
except ImportError:
    pass
else:
    import os, sys, subprocess
    drive.mount("/content/drive")
    repo = Path("/content/longitudinal-aging-diffusion")
    if not repo.is_dir():
        subprocess.check_call(["git", "clone", "https://github.com/pablo-reyes8/longitudinal-aging-diffusion", str(repo)])
    os.chdir(repo)
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[aging-bias,evaluation,comparison,auxiliary]"])

from src.comparisong.long_gap_experiment import run_long_gap_experiment

result = run_long_gap_experiment(
    train_root=TRAIN_ROOT,
    reference_dir=REFERENCE_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    output_dir=OUTPUT,
    models=MODELS,
    **CONFIG,
)

### 4. Análisis
Residuo positivo por zona: más cambio que el esperado en módulo 1; negativo: menos. Las tablas completas quedan en `OUTPUT`.

In [ ]:
from src.comparisong.long_gap_experiment import display_long_gap_analysis

display_long_gap_analysis(result)